In [39]:
import torch
from torch.utils.data import DataLoader
from datasets import load_dataset

# 1. Load the IMDB dataset from Hugging Face

raw_datasets = load_dataset("stanfordnlp/imdb")
raw_datasets.shape



{'train': (25000, 2), 'test': (25000, 2), 'unsupervised': (50000, 2)}

In [40]:
data = torch.load("text8_embeddings.pt")

pretrained_embeddings = data["embeddings"]
word_to_key = data["word_to_id"]
vocab = data["vocab"]

vocab_size = len(vocab)

print(pretrained_embeddings.shape)
print(len(word_to_key))
print(vocab_size)

torch.Size([209174, 100])
209174
209174


In [41]:

embedding_dim = pretrained_embeddings.shape[1]

PAD_ID = vocab_size
UNK_ID = vocab_size + 1

special_embeddings = torch.randn(2,embedding_dim) * 0.1

embedding_matrix = torch.cat(
    [pretrained_embeddings, special_embeddings],
    dim=0
)

print(embedding_matrix.shape)

torch.Size([209176, 100])


In [42]:
import re

def encode_review(text):

    words = re.findall(
        r"\b\w+\b",
        text.lower()
    )

    ids = [
        word_to_key.get(word, UNK_ID)
        for word in words
    ]

    return ids

def encode_examples(examples):

    input_ids = [
        encode_review(text)
        for text in examples["text"]
    ]

    return {
        "input_ids": input_ids,
        "labels": examples["label"]
    }

tokenized_datasets = raw_datasets.map(
    encode_examples,
    batched=True,
    remove_columns=["text"]
)

In [43]:
from torch.nn.utils.rnn import pad_sequence


def collate_fn(batch):

    input_ids = [
        torch.tensor(
            item["input_ids"],
            dtype=torch.long
        )
        for item in batch
    ]

    labels = torch.tensor(
        [item["labels"] for item in batch],
        dtype=torch.float
    )

    lengths = torch.tensor(
        [len(x) for x in input_ids],
        dtype=torch.long
    )

    input_ids = pad_sequence(
        input_ids,
        batch_first=True,
        padding_value=PAD_ID
    )

    attention_mask = (
        torch.arange(input_ids.size(1))[None, :]
        < lengths[:, None]
    )

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
        "lengths": lengths
    }

In [44]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(
    tokenized_datasets["train"],
    shuffle=True,
    batch_size=32,
    collate_fn=collate_fn,
    pin_memory=torch.cuda.is_available()
)

test_dataloader = DataLoader(
    tokenized_datasets["test"],
    batch_size=32,
    collate_fn=collate_fn,
    pin_memory=torch.cuda.is_available()
)


In [45]:
from torch.nn.utils.rnn import pack_padded_sequence

class IMDBClassifier(torch.nn.Module):

    def __init__(self,embedding_matrix,freeze_embeddings=True, hidden_dim = 128,num_layers =1,dropout = 0.2):
        super().__init__()

        self.embedding = torch.nn.Embedding.from_pretrained(
            embedding_matrix,
            freeze=freeze_embeddings,
            padding_idx=PAD_ID
        )

        embedding_dim = embedding_matrix.shape[1]

        self.LSTM = torch.nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        self.classifier = torch.nn.Linear(
            hidden_dim,
            1
        )

    def forward(self,input_ids,lengths):

        embeddings = self.embedding(input_ids)

        packed = pack_padded_sequence(
            embeddings,
            lengths,
            batch_first=True,
            enforce_sorted=False
        )

        outputs, (hidden, cell) = self.LSTM(packed)

        final_hidden = hidden[-1]

        logits = self.classifier(final_hidden)

        return logits.squeeze(1)


In [46]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = IMDBClassifier(
    embedding_matrix=embedding_matrix,
    freeze_embeddings=True,
    hidden_dim=128
).to(device)

print(model)

IMDBClassifier(
  (embedding): Embedding(209176, 100, padding_idx=209174)
  (LSTM): LSTM(100, 128, batch_first=True)
  (classifier): Linear(in_features=128, out_features=1, bias=True)
)


In [47]:
criterion = torch.nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

In [49]:
num_epochs = 5

for epoch in range(num_epochs):

    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for batch in train_dataloader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        lengths = batch["lengths"]
        
        optimizer.zero_grad()

        logits = model(
            input_ids,
            lengths
        )

        loss = criterion(
            logits,
            labels
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

        predictions = (
            torch.sigmoid(logits) >= 0.5
        ).float()

        correct += (
            predictions == labels
        ).sum().item()

        total += labels.size(0)

    train_loss = total_loss / len(train_dataloader)
    train_accuracy = correct / total

    print(
        f"Epoch {epoch + 1}/{num_epochs} "
        f"| Loss: {train_loss:.4f} "
        f"| Accuracy: {train_accuracy:.4f}"
    )

Epoch 1/5 | Loss: 0.6558 | Accuracy: 0.6022
Epoch 2/5 | Loss: 0.6260 | Accuracy: 0.6538
Epoch 3/5 | Loss: 0.5838 | Accuracy: 0.6842
Epoch 4/5 | Loss: 0.5149 | Accuracy: 0.7440
Epoch 5/5 | Loss: 0.4346 | Accuracy: 0.8022


In [50]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for batch in test_dataloader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        lengths = batch["lengths"]

        logits = model(
            input_ids,
            lengths
        )

        predictions = (
            torch.sigmoid(logits) >= 0.5
        ).float()

        correct += (
            predictions == labels
        ).sum().item()

        total += labels.size(0)

test_accuracy = correct / total

print(f"Test accuracy: {test_accuracy:.4f}")

Test accuracy: 0.7914
